# Translation
### AI translates English to Tamil and more!
---
**Model:** facebook/nllb-200-distilled-600M

> NLLB = No Language Left Behind — supports 200 languages!
> We load this model directly (not via pipeline — NLLB needs special handling)


---
## Step 1 — Install and load


In [ ]:
!pip install transformers sentencepiece gradio -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

# Load model and tokenizer separately
# (NLLB does not work with pipeline — we load it directly)
MODEL_NAME = 'facebook/nllb-200-distilled-600M'

print('Loading model... takes about 2 minutes')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('Model ready!')


---
## Step 2 — Create a simple translate function

> This function takes English text and a target language code.
> Returns the translated text.


In [ ]:
def translate(text, tgt_lang):
    # Step A: tell tokenizer the source language is English
    tokenizer.src_lang = 'eng_Latn'

    # Step B: convert text into numbers the model understands
    inputs = tokenizer(text, return_tensors='pt')

    # Step C: get the ID number for the target language
    target_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    # Step D: generate the translation
    output = model.generate(
        **inputs,
        forced_bos_token_id = target_id,
        max_new_tokens      = 200
    )

    # Step E: convert numbers back to text
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    return result

print('translate() function ready!')


---
## Step 3 — Translate one sentence


In [ ]:
sentence = 'Artificial intelligence is changing the world.'

result = translate(sentence, tgt_lang='tam_Taml')

print('English :', sentence)
print('Tamil   :', result)


---
## Step 4 — Translate multiple sentences


In [ ]:
sentences = [
    'Good morning! How are you?',
    'AI is the future of technology.',
    'Smart cities use sensors and data.',
    'Students are learning about machine learning.',
]

print('English → Tamil:')
print()
for sentence in sentences:
    result = translate(sentence, tgt_lang='tam_Taml')
    print(f'English : {sentence}')
    print(f'Tamil   : {result}')
    print()


---
## Step 5 — Try your own sentence!

> ✏️ Change the sentence below to anything in English!


In [ ]:
# ✏️ Change this to any English sentence!
my_sentence = 'I am learning about artificial intelligence today.'

result = translate(my_sentence, tgt_lang='tam_Taml')

print('English :', my_sentence)
print('Tamil   :', result)


---
## Step 6 — Translate to other languages

| Language | Code to use |
|----------|-------------|
| Tamil | `tam_Taml` |
| Hindi | `hin_Deva` |
| French | `fra_Latn` |
| Spanish | `spa_Latn` |
| Arabic | `arb_Arab` |
| Telugu | `tel_Telu` |
| Malayalam | `mal_Mlym` |


In [ ]:
sentence = 'Technology is changing our lives every day.'

languages = [
    ('Tamil',     'tam_Taml'),
    ('Hindi',     'hin_Deva'),
    ('French',    'fra_Latn'),
    ('Spanish',   'spa_Latn'),
    ('Telugu',    'tel_Telu'),
    ('Malayalam', 'mal_Mlym'),
]

print(f'Original: {sentence}')
print()
for lang_name, lang_code in languages:
    result = translate(sentence, tgt_lang=lang_code)
    print(f'{lang_name:10s}: {result}')


---
## Step 7 — Build a Gradio App

> Type any English sentence.
> Pick a language from the dropdown.
> AI translates instantly!


In [ ]:
import gradio as gr

# Language options for the dropdown
language_map = {
    'Tamil'    : 'tam_Taml',
    'Hindi'    : 'hin_Deva',
    'French'   : 'fra_Latn',
    'Spanish'  : 'spa_Latn',
    'Arabic'   : 'arb_Arab',
    'Telugu'   : 'tel_Telu',
    'Malayalam': 'mal_Mlym',
}

def translate_app(sentence, language):
    if not sentence.strip():
        return 'Please type a sentence first!'

    lang_code = language_map[language]
    result    = translate(sentence, tgt_lang=lang_code)
    return result


app = gr.Interface(
    fn          = translate_app,
    inputs      = [
        gr.Textbox(
            label       = 'English sentence',
            placeholder = 'Type any English sentence...',
            lines       = 2
        ),
        gr.Dropdown(
            choices = list(language_map.keys()),
            value   = 'Tamil',
            label   = 'Translate to'
        )
    ],
    outputs     = gr.Textbox(label='Translation', lines=3),
    title       = 'AI Translator — NLLB 200 Languages',
    description = 'Type English → pick a language → AI translates!'
)

app.launch(share=True)


In [ ]:
app.close()
print('App closed.')


---
## What we learned
- NLLB model is loaded directly using `AutoTokenizer` and `AutoModelForSeq2SeqLM`
- We wrote a simple `translate()` function that handles everything inside
- Change `tgt_lang` code to switch to any of 200 languages
- Gradio dropdown lets users pick the language at runtime

**Tomorrow — Day 8:** We build a chatbot that remembers your conversation!
